# 10 — Prepare inferCNV / infercnvpy inputs on the canonical article branch

This notebook reproduces the archived CNV-input preparation step on the
canonical 18-cluster branch while avoiding the historical full-object
`sc.read_h5ad()` / `.copy()` path that is unsafe on the current Windows /
16-GB-RAM system.

## Historical scientific semantics preserved

The archived executed analysis used:

- 351,427 cells;
- 15,176 full-gene genes;
- canonical `leiden_scvi_main`, `r = 0.60`, 18 clusters;
- raw non-negative integer counts from `layers["counts"]`;
- GENCODE v49 primary-assembly gene coordinates;
- 15,163 autosomal/X/Y genes for chromosome-ordered CNV input;
- 13 mitochondrial genes excluded from the chromosome-ordered object;
- the same five CNV reference/query strategies;
- the same random seed and per-cluster sampling limits.

The final DS1/DS2/DS3/DS4 nomenclature is retained. The archived free-text note
that described lymphoid references as `DS5`-biased is updated to `DS3`-biased;
this is a naming update only.

## Memory-safe implementation

The large raw count matrix is read sequentially from HDF5. In one resumable
streaming pass the notebook writes:

1. an all-cell full-gene count-X H5AD;
2. an all-cell chromosome-ordered count-X H5AD.

Both are built under NVMe scratch and then copied sequentially to the
repository. Strategy-specific test subsets are extracted one at a time from the
NVMe chromosome-ordered object, so only one sampled sparse matrix is in RAM at
a time.

No 351,427 × 15,176 AnnData count matrix is materialized in RAM.

## Archived execution checkpoints

```text
readiness report                         (1, 15)
strategy definitions                     (5, 6)
cluster category counts                 (18, 7)
cell category counts                     (9, 3)

GENCODE unique symbols               77,271
matched genes                        15,176
canonical usable genes               15,163
unmatched/noncanonical genes             13

all-cell full-gene H5AD          351427 × 15176
all-cell gene-ordered H5AD       351427 × 15163

strategy manifest                       (5, 12)
reference-group summary                  (5, 3)
strategy composition                    (10, 20)
final manifest                          (23, 4)
```

Historical sampled test-subset sizes:

```text
core_conservative      32,640
core_plus_lymphoid     38,807
extended_myeloid       45,000
review_cycling         14,640
review_ambiguous       19,443
```

The same historical sampling algorithm and seed are used. Because the canonical
repository object is stored in the final standardized cell order rather than
the old pre-renaming row order, sampled **sizes and strategy semantics** are
hard-checked; sampled cell identities are deterministic for the canonical
repository object and are not falsely claimed to be byte-for-byte identical to
the archived sampled test objects.

In [1]:
# =========================
# 0) IMPORTS / PATHS / PARAMETERS
# =========================

from pathlib import Path

import os
import re
import gc
import gzip
import json
import hashlib
import shutil
import warnings

import h5py
import numpy as np
import pandas as pd
import scipy.sparse as sp

import scanpy as sc
import anndata as ad

try:
    from anndata.io import read_elem
except ImportError:
    from anndata._io.specs import read_elem


warnings.filterwarnings("ignore")


def _find_repository_root(start=None):
    start = (
        Path.cwd()
        if start is None
        else Path(start)
    )
    start = start.expanduser().resolve()

    for candidate in (
        start,
        *start.parents,
    ):
        if (
            (candidate / "README.md").exists()
            and (candidate / "code").exists()
        ):
            return candidate

    return start


PROJECT_DIR = Path(
    os.environ.get(
        "GLIOMA_PROJECT_ROOT",
        _find_repository_root(),
    )
).expanduser().resolve()

PROCESSED_DIR = (
    PROJECT_DIR
    / "data"
    / "processed"
)

RAW_DIR = (
    PROJECT_DIR
    / "data"
    / "interim"
)

METADATA_DIR = (
    PROJECT_DIR
    / "metadata"
)

FIGURES_DIR = (
    PROJECT_DIR
    / "figures"
)

REFERENCE_DIR = (
    PROJECT_DIR
    / "data"
    / "reference"
)

CNV_DIR = (
    RAW_DIR
    / "integrated"
    / "CNV_inputs"
)

for directory in (
    PROCESSED_DIR,
    RAW_DIR,
    METADATA_DIR,
    FIGURES_DIR,
    REFERENCE_DIR,
    CNV_DIR,
):
    directory.mkdir(
        parents=True,
        exist_ok=True,
    )


# Notebook-09 canonical bias-ready object.
INPUT_H5AD = (
    PROCESSED_DIR
    / "integrated"
    / "GBM_core_4datasets_fullgene_marker_based_annotation_bias_status.h5ad"
)

FALLBACK_H5AD = (
    PROCESSED_DIR
    / "integrated"
    / "GBM_core_4datasets_fullgene_marker_based_annotation.h5ad"
)

GTF_GZ = (
    REFERENCE_DIR
    / "gencode.v49.primary_assembly.annotation.gtf.gz"
)


CLUSTER_KEY = "leiden_scvi_main"
ANNOTATION_COL = "annotation_marker_based_HVG8000_r0_6"
LINEAGE_COL = "lineage_marker_based_HVG8000_r0_6"
CONFIDENCE_COL = "annotation_confidence_HVG8000_r0_6"
CNV_REQUIRED_COL = "requires_CNV_for_malignancy_call_HVG8000_r0_6"
BIAS_STATUS_COL = "cluster_bias_status_HVG8000_r0_6"
BIAS_REASON_COL = "cluster_bias_reason_HVG8000_r0_6"

# Historical notebook checked this old optional name; it was absent there.
CNV_ROLE_COL = "cnv_role_recommendation_HVG8000_r0_6"

CNV_CLUSTER_ROLE_COL = "cnv_cluster_role_HVG8000_r0_6"


RANDOM_SEED = 42
np.random.seed(
    RANDOM_SEED
)

MAX_REF_PER_STRATEGY = 20_000
MAX_QUERY_PER_STRATEGY = 25_000
MAX_PER_CLUSTER_REF = 5_000
MAX_PER_CLUSTER_QUERY = 7_000


EXPECTED_N_CELLS = 351_427
EXPECTED_N_GENES = 15_176
EXPECTED_GENE_ORDERED_GENES = 15_163
EXPECTED_N_CLUSTERS = 18

EXPECTED_CLUSTER_MIN = 105
EXPECTED_CLUSTER_MAX = 61_646
EXPECTED_CLUSTER_MEDIAN = 14_336.5

EXPECTED_GTF_UNIQUE_SYMBOLS = 77_271
EXPECTED_UNMATCHED_GENES = 13

EXPECTED_READINESS_SHAPE = (1, 15)
EXPECTED_STRATEGY_SHAPE = (5, 6)
EXPECTED_CLUSTER_CATEGORY_SHAPE = (18, 7)
EXPECTED_CELL_CATEGORY_SHAPE = (9, 3)
EXPECTED_POSITION_REPORT_SHAPE = (1, 6)
EXPECTED_GENE_ORDER_SHAPE = (15_163, 4)
EXPECTED_UNMATCHED_SHAPE = (13, 6)
EXPECTED_STRATEGY_MANIFEST_SHAPE = (5, 12)
EXPECTED_REF_GROUPS_SHAPE = (5, 3)
EXPECTED_COMPOSITION_SHAPE = (10, 20)
EXPECTED_MANIFEST_SHAPE = (23, 4)


# Historical strategy/test-subset row counts.
EXPECTED_STRATEGY_COUNTS = {
    "core_conservative": {
        "annotation": 140_699,
        "reference_all": 17_449,
        "query_all": 123_250,
        "sampled_reference": 7_640,
        "sampled_query": 25_000,
        "sampled_total": 32_640,
    },
    "core_plus_lymphoid": {
        "annotation": 165_843,
        "reference_all": 42_593,
        "query_all": 123_250,
        "sampled_reference": 13_807,
        "sampled_query": 25_000,
        "sampled_total": 38_807,
    },
    "extended_myeloid": {
        "annotation": 258_788,
        "reference_all": 135_538,
        "query_all": 123_250,
        "sampled_reference": 20_000,
        "sampled_query": 25_000,
        "sampled_total": 45_000,
    },
    "review_cycling": {
        "annotation": 56_083,
        "reference_all": 17_449,
        "query_all": 38_634,
        "sampled_reference": 7_640,
        "sampled_query": 7_000,
        "sampled_total": 14_640,
    },
    "review_ambiguous": {
        "annotation": 36_116,
        "reference_all": 17_449,
        "query_all": 18_667,
        "sampled_reference": 7_640,
        "sampled_query": 11_803,
        "sampled_total": 19_443,
    },
}


EXPECTED_ROLE_CELL_COUNTS = {
    "query_putative_tumor_like": 123_250,
    "reference_myeloid_extended_candidate": 118_089,
    "query_review_cycling": 38_634,
    "reference_lymphoid_candidate": 25_144,
    "query_review_ambiguous": 18_667,
    "reference_core_conservative": 17_449,
    "reference_myeloid_review_not_core": 6_340,
    "reference_stromal_review_not_core": 3_566,
    "exclude_or_low_confidence": 288,
}


# Large final outputs.
ALLCELLS_H5AD = (
    CNV_DIR
    / "GBM_core_4datasets_CNV_input_HVG8000_r0_6_allcells_fullgene_countsX.h5ad"
)

GENE_ORDERED_H5AD = (
    CNV_DIR
    / "GBM_core_4datasets_CNV_input_HVG8000_r0_6_allcells_gene_ordered_countsX.h5ad"
)


def _default_scratch_root():
    configured = os.environ.get(
        "GLIOMA_MAIN10_SCRATCH_ROOT"
    )

    if configured:
        return (
            Path(configured)
            .expanduser()
            .resolve()
        )

    if (
        os.name == "nt"
        and Path("E:/").exists()
    ):
        return Path(
            r"E:\glioma-cnv-scratch\main_analysis_10"
        )

    return (
        PROJECT_DIR
        / ".scratch"
        / "main_analysis_10"
    ).resolve()


SCRATCH_ROOT = _default_scratch_root()
SCRATCH_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)

SCRATCH_ALLCELLS_H5AD = (
    SCRATCH_ROOT
    / ALLCELLS_H5AD.name
)

SCRATCH_GENE_ORDERED_H5AD = (
    SCRATCH_ROOT
    / GENE_ORDERED_H5AD.name
)

STREAM_PROGRESS_JSON = (
    SCRATCH_ROOT
    / "cnv_input_stream_progress.json"
)


# Sequential source-read target.
MAX_NNZ_PER_ROW_BLOCK = 8_000_000


def write_tsv_replace(
    dataframe,
    path,
):
    path = Path(path)
    path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    if path.exists():
        path.unlink()

    dataframe.to_csv(
        path,
        sep="\t",
        index=False,
    )

    print(
        f"[SAVED] {path} "
        f"shape={dataframe.shape}"
    )


def write_text_replace(
    lines,
    path,
):
    path = Path(path)
    path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    if path.exists():
        path.unlink()

    with path.open(
        "w",
        encoding="utf-8",
    ) as handle:
        for line in lines:
            handle.write(
                str(line).rstrip(
                    "\n"
                )
                + "\n"
            )

    print(
        f"[SAVED] {path} "
        f"n_lines={len(lines)}"
    )


def write_json_atomic(
    payload,
    path,
):
    path = Path(path)
    temporary = path.with_suffix(
        path.suffix + ".tmp"
    )

    with temporary.open(
        "w",
        encoding="utf-8",
    ) as handle:
        json.dump(
            payload,
            handle,
            indent=2,
            sort_keys=True,
        )

    os.replace(
        temporary,
        path,
    )


def hash_strings(
    values,
):
    digest = hashlib.sha256()

    for value in values:
        digest.update(
            str(value).encode(
                "utf-8"
            )
        )
        digest.update(
            b"\0"
        )

    return digest.hexdigest()


def sample_cells_by_cluster(
    obs_frame,
    clusters,
    cluster_col,
    max_total,
    max_per_cluster,
    seed=42,
):
    """
    Exact historical sampling algorithm.
    """
    rng = np.random.default_rng(
        seed
    )

    chosen = []

    obs2 = obs_frame[
        obs_frame[
            cluster_col
        ]
        .astype(str)
        .isin(
            [
                str(cluster)
                for cluster
                in clusters
            ]
        )
    ]

    for cluster in sorted(
        obs2[
            cluster_col
        ]
        .astype(str)
        .unique(),
        key=lambda value:
            int(value)
            if str(value).isdigit()
            else str(value),
    ):
        index = (
            obs2.index[
                obs2[
                    cluster_col
                ]
                .astype(str)
                == str(cluster)
            ]
            .to_numpy()
        )

        n_select = min(
            len(index),
            max_per_cluster,
        )

        if n_select > 0:
            chosen.extend(
                rng.choice(
                    index,
                    size=n_select,
                    replace=False,
                ).tolist()
            )

    if len(chosen) > max_total:
        chosen = rng.choice(
            np.asarray(
                chosen
            ),
            size=max_total,
            replace=False,
        ).tolist()

    return list(
        chosen
    )


def parse_gtf_gene_positions(
    gtf_gz_path,
):
    rows = []

    if not Path(
        gtf_gz_path
    ).exists():
        raise FileNotFoundError(
            f"GTF not found: {gtf_gz_path}"
        )

    opener = (
        gzip.open
        if str(
            gtf_gz_path
        ).endswith(
            ".gz"
        )
        else open
    )

    with opener(
        gtf_gz_path,
        "rt",
    ) as handle:
        for line in handle:
            if (
                not line
                or line.startswith(
                    "#"
                )
            ):
                continue

            parts = (
                line.rstrip(
                    "\n"
                )
                .split(
                    "\t"
                )
            )

            if len(
                parts
            ) < 9:
                continue

            (
                chrom,
                source,
                feature,
                start,
                end,
                score,
                strand,
                frame,
                attrs,
            ) = parts

            if feature != "gene":
                continue

            gene_id = None
            gene_name = None
            gene_type = None

            for match in re.finditer(
                r'(gene_id|gene_name|gene_type|gene_biotype) "([^"]+)"',
                attrs,
            ):
                key, value = (
                    match.group(
                        1
                    ),
                    match.group(
                        2
                    ),
                )

                if key == "gene_id":
                    gene_id = (
                        value.split(
                            "."
                        )[0]
                    )

                elif key == "gene_name":
                    gene_name = value

                elif key in (
                    "gene_type",
                    "gene_biotype",
                ):
                    gene_type = value

            if gene_name is None:
                continue

            rows.append(
                {
                    "gene_symbol":
                        gene_name,
                    "gene_id":
                        gene_id,
                    "chromosome":
                        chrom.replace(
                            "chr",
                            "",
                        ),
                    "start":
                        int(
                            start
                        ),
                    "end":
                        int(
                            end
                        ),
                    "strand":
                        strand,
                    "gene_type":
                        gene_type,
                }
            )

    gtf = pd.DataFrame(
        rows
    )

    chrom_order = {
        str(index):
            index
        for index
        in range(
            1,
            23,
        )
    }

    chrom_order.update(
        {
            "X": 23,
            "Y": 24,
            "M": 25,
            "MT": 25,
        }
    )

    gtf[
        "chrom_order"
    ] = (
        gtf[
            "chromosome"
        ]
        .map(
            chrom_order
        )
        .fillna(
            999
        )
        .astype(
            int
        )
    )

    gtf[
        "protein_coding_rank"
    ] = (
        gtf[
            "gene_type"
        ]
        == "protein_coding"
    ).astype(
        int
    )

    gtf = gtf.sort_values(
        [
            "gene_symbol",
            "protein_coding_rank",
            "chrom_order",
            "start",
        ],
        ascending=[
            True,
            False,
            True,
            True,
        ],
    )

    gtf = gtf.drop_duplicates(
        "gene_symbol",
        keep="first",
    ).copy()

    return gtf


print(
    "Project:",
    PROJECT_DIR,
)

print(
    "Input h5ad:",
    INPUT_H5AD,
)

print(
    "GTF:",
    GTF_GZ,
)

print(
    "CNV output dir:",
    CNV_DIR,
)

print(
    "Scratch root:",
    SCRATCH_ROOT,
)

Project: G:\Repository\glioma-cnv-single-cell-atlas
Input h5ad: G:\Repository\glioma-cnv-single-cell-atlas\data\processed\integrated\GBM_core_4datasets_fullgene_marker_based_annotation_bias_status.h5ad
GTF: G:\Repository\glioma-cnv-single-cell-atlas\data\reference\gencode.v49.primary_assembly.annotation.gtf.gz
CNV output dir: G:\Repository\glioma-cnv-single-cell-atlas\data\interim\integrated\CNV_inputs
Scratch root: E:\glioma-cnv-scratch\main_analysis_10


In [2]:
# =========================
# 1) METADATA-ONLY INPUT VALIDATION
# =========================

if INPUT_H5AD.exists():
    input_path = INPUT_H5AD

elif FALLBACK_H5AD.exists():
    input_path = FALLBACK_H5AD

    print(
        "[WARNING] Bias-status object not found. "
        "Using fallback annotated object:",
        input_path,
    )

else:
    raise FileNotFoundError(
        "Neither canonical CNV input exists:\n"
        f"{INPUT_H5AD}\n"
        f"{FALLBACK_H5AD}"
    )


with h5py.File(
    input_path,
    "r",
) as handle:

    if "obs" not in handle:
        raise KeyError(
            "Input H5AD is missing /obs."
        )

    if "var" not in handle:
        raise KeyError(
            "Input H5AD is missing /var."
        )

    if (
        "layers" not in handle
        or "counts"
        not in handle[
            "layers"
        ]
    ):
        raise KeyError(
            "CNV preparation requires layers['counts']."
        )

    obs = read_elem(
        handle[
            "obs"
        ]
    )

    var = read_elem(
        handle[
            "var"
        ]
    )

    counts = (
        handle[
            "layers"
        ][
            "counts"
        ]
    )

    counts_encoding = (
        counts.attrs.get(
            "encoding-type",
            "",
        )
    )

    if isinstance(
        counts_encoding,
        bytes,
    ):
        counts_encoding = (
            counts_encoding.decode(
                "utf-8"
            )
        )

    counts_shape = tuple(
        int(value)
        for value
        in counts.attrs[
            "shape"
        ]
    )

    counts_nnz = int(
        counts[
            "data"
        ].shape[
            0
        ]
    )

    counts_data_dtype = str(
        counts[
            "data"
        ].dtype
    )

    counts_indices_dtype = str(
        counts[
            "indices"
        ].dtype
    )

    counts_indptr_dtype = str(
        counts[
            "indptr"
        ].dtype
    )

    source_indptr = np.asarray(
        counts[
            "indptr"
        ],
        dtype=np.int64,
    )


    # Reproduce the archived sparse_integer_like sample:
    # first 2000 rows × first 2000 genes.
    sample_rows = min(
        2_000,
        counts_shape[
            0
        ],
    )

    sample_stop = int(
        source_indptr[
            sample_rows
        ]
    )

    sample_values = np.asarray(
        counts[
            "data"
        ][
            :sample_stop
        ]
    )

    sample_indices = np.asarray(
        counts[
            "indices"
        ][
            :sample_stop
        ],
        dtype=np.int32,
    )

    sample_values = sample_values[
        sample_indices
        < min(
            2_000,
            counts_shape[
                1
            ],
        )
    ]


    x_scvi = (
        np.asarray(
            handle[
                "obsm"
            ][
                "X_scVI"
            ],
            dtype=np.float32,
        )
        if (
            "obsm" in handle
            and "X_scVI"
            in handle[
                "obsm"
            ]
        )
        else None
    )

    x_umap = (
        np.asarray(
            handle[
                "obsm"
            ][
                "X_umap"
            ],
            dtype=np.float32,
        )
        if (
            "obsm" in handle
            and "X_umap"
            in handle[
                "obsm"
            ]
        )
        else None
    )


if counts_encoding != "csr_matrix":
    raise AssertionError(
        "Raw count layer is not CSR."
    )

if counts_shape != (
    EXPECTED_N_CELLS,
    EXPECTED_N_GENES,
):
    raise AssertionError(
        "Unexpected raw-count shape: "
        f"{counts_shape}"
    )

if len(
    obs
) != EXPECTED_N_CELLS:
    raise AssertionError(
        "Unexpected cell count."
    )

if len(
    var
) != EXPECTED_N_GENES:
    raise AssertionError(
        "Unexpected gene count."
    )

if not var.index.is_unique:
    raise AssertionError(
        "Full-gene var_names are not unique."
    )


required_obs = [
    CLUSTER_KEY,
    ANNOTATION_COL,
    LINEAGE_COL,
    CONFIDENCE_COL,
]

missing_required = [
    column
    for column
    in required_obs
    if column
    not in obs.columns
]

if missing_required:
    raise KeyError(
        "Missing required obs columns: "
        f"{missing_required}"
    )


string_columns = [
    CLUSTER_KEY,
    ANNOTATION_COL,
    LINEAGE_COL,
    CONFIDENCE_COL,
    CNV_REQUIRED_COL,
    BIAS_STATUS_COL,
    BIAS_REASON_COL,
    CNV_ROLE_COL,
    "core_dataset",
    "dataset_id",
    "condition",
    "patient_id",
    "sample_id",
    "scvi_batch",
]

for column in string_columns:
    if column in obs.columns:
        obs[
            column
        ] = (
            obs[
                column
            ]
            .astype(str)
            .replace(
                {
                    "nan":
                        "unknown",
                    "None":
                        "unknown",
                    "<NA>":
                        "unknown",
                }
            )
        )


cluster_counts = (
    obs[
        CLUSTER_KEY
    ]
    .astype(str)
    .value_counts()
)


if cluster_counts.shape[0] != EXPECTED_N_CLUSTERS:
    raise AssertionError(
        "Canonical input is not 18 clusters."
    )

if int(
    cluster_counts.min()
) != EXPECTED_CLUSTER_MIN:
    raise AssertionError(
        "Canonical minimum cluster size changed."
    )

if int(
    cluster_counts.max()
) != EXPECTED_CLUSTER_MAX:
    raise AssertionError(
        "Canonical maximum cluster size changed."
    )

if not np.isclose(
    float(
        cluster_counts.median()
    ),
    EXPECTED_CLUSTER_MEDIAN,
):
    raise AssertionError(
        "Canonical median cluster size changed."
    )


expected_datasets = {
    "DS1_GSE103224",
    "DS2_GSE141383",
    "DS3_GSE182109",
    "DS4_GSE278450",
}

observed_datasets = set(
    obs[
        "core_dataset"
    ]
    .astype(str)
    .unique()
)

if observed_datasets != expected_datasets:
    raise AssertionError(
        "Input core_dataset nomenclature is not "
        "the final DS1/DS2/DS3/DS4 scheme."
    )


if sample_values.size == 0:
    integer_like = True
    x_min = None
    x_max = None
    any_negative = False

else:
    finite_values = sample_values[
        np.isfinite(
            sample_values
        )
    ]

    integer_like = bool(
        np.allclose(
            finite_values,
            np.round(
                finite_values
            ),
        )
    )

    x_min = float(
        finite_values.min()
    )

    x_max = float(
        finite_values.max()
    )

    any_negative = bool(
        np.any(
            finite_values
            < 0
        )
    )


print(
    "counts_source:",
    "adata.layers['counts']",
)

print(
    "X integer-like:",
    integer_like,
)

print(
    "X min/max:",
    x_min,
    x_max,
)

print(
    "any negative:",
    any_negative,
)


if not integer_like or any_negative:
    raise ValueError(
        "CNV input requires raw non-negative "
        "integer-like counts."
    )


# These two values are archived execution checkpoints.
if not np.isclose(
    x_min,
    1.0,
):
    raise AssertionError(
        f"Historical count-sample minimum changed: {x_min}"
    )

if not np.isclose(
    x_max,
    389.0,
):
    raise AssertionError(
        f"Historical count-sample maximum changed: {x_max}"
    )


readiness = pd.DataFrame(
    [
        {
            "input_h5ad":
                str(
                    input_path
                ),

            "n_cells":
                EXPECTED_N_CELLS,

            "n_genes":
                EXPECTED_N_GENES,

            "counts_source_for_cnv":
                "adata.layers['counts']",

            "has_counts_layer":
                True,

            "x_integer_like":
                integer_like,

            "x_min_sample":
                x_min,

            "x_max_sample":
                x_max,

            "any_negative_sample":
                any_negative,

            "cluster_key":
                CLUSTER_KEY,

            "annotation_col":
                ANNOTATION_COL,

            "lineage_col":
                LINEAGE_COL,

            "confidence_col":
                CONFIDENCE_COL,

            "bias_status_col_present":
                BIAS_STATUS_COL
                in obs.columns,

            "cnv_role_col_present":
                CNV_ROLE_COL
                in obs.columns,
        }
    ]
)


READINESS_TSV = (
    METADATA_DIR
    / "CNV_input_readiness_report_HVG8000_r0_6.tsv"
)

write_tsv_replace(
    readiness,
    READINESS_TSV,
)

display(
    readiness
)


if readiness.shape != EXPECTED_READINESS_SHAPE:
    raise AssertionError(
        "Historical readiness shape changed."
    )


if bool(
    readiness.iloc[
        0
    ][
        "bias_status_col_present"
    ]
) is not True:
    raise AssertionError(
        "Historical bias-status readiness checkpoint failed."
    )


print(
    "\n[CANONICAL CNV INPUT VALIDATED]"
)

counts_source: adata.layers['counts']
X integer-like: True
X min/max: 1.0 389.0
any negative: False
[SAVED] G:\Repository\glioma-cnv-single-cell-atlas\metadata\CNV_input_readiness_report_HVG8000_r0_6.tsv shape=(1, 15)


,input_h5ad,n_cells,n_genes,counts_source_for_cnv,has_counts_layer,x_integer_like,x_min_sample,x_max_sample,any_negative_sample,cluster_key,annotation_col,lineage_col,confidence_col,bias_status_col_present,cnv_role_col_present
0,G:\Repository\glioma-cnv-single-cell-atlas\dat...,351427,15176,adata.layers['counts'],True,True,1.0,389.0,False,leiden_scvi_main,annotation_marker_based_HVG8000_r0_6,lineage_marker_based_HVG8000_r0_6,annotation_confidence_HVG8000_r0_6,True,False



[CANONICAL CNV INPUT VALIDATED]


In [3]:
# =========================
# 2) CNV CLUSTER ROLES / STRATEGIES / HISTORICAL COUNTS
# =========================

query_tumor_like = [
    "0",
    "4",
    "7",
    "11",
]

review_cycling = [
    "2",
]

review_ambiguous = [
    "9",
    "12",
]

exclude_or_low_conf = [
    "16",
    "17",
]

core_reference = [
    "8",
    "14",
]

lymphoid_reference = [
    "5",
    "15",
]

myeloid_reference = [
    "1",
    "3",
    "6",
]

myeloid_review = [
    "10",
]

stromal_review = [
    "13",
]


role_map = {}

for cluster in query_tumor_like:
    role_map[
        cluster
    ] = (
        "query_putative_tumor_like"
    )

for cluster in review_cycling:
    role_map[
        cluster
    ] = (
        "query_review_cycling"
    )

for cluster in review_ambiguous:
    role_map[
        cluster
    ] = (
        "query_review_ambiguous"
    )

for cluster in exclude_or_low_conf:
    role_map[
        cluster
    ] = (
        "exclude_or_low_confidence"
    )

for cluster in core_reference:
    role_map[
        cluster
    ] = (
        "reference_core_conservative"
    )

for cluster in lymphoid_reference:
    role_map[
        cluster
    ] = (
        "reference_lymphoid_candidate"
    )

for cluster in myeloid_reference:
    role_map[
        cluster
    ] = (
        "reference_myeloid_extended_candidate"
    )

for cluster in myeloid_review:
    role_map[
        cluster
    ] = (
        "reference_myeloid_review_not_core"
    )

for cluster in stromal_review:
    role_map[
        cluster
    ] = (
        "reference_stromal_review_not_core"
    )


obs[
    CNV_CLUSTER_ROLE_COL
] = (
    obs[
        CLUSTER_KEY
    ]
    .astype(str)
    .map(
        role_map
    )
    .fillna(
        "unassigned_review"
    )
    .astype(
        "category"
    )
)


strategies = {
    "core_conservative": {
        "reference_clusters":
            core_reference,

        "query_clusters":
            query_tumor_like,

        "description":
            (
                "Most conservative CNV run: oligodendrocyte "
                "and endothelial references only; putative "
                "tumor-like query clusters only."
            ),
    },

    "core_plus_lymphoid": {
        "reference_clusters":
            core_reference
            + lymphoid_reference,

        "query_clusters":
            query_tumor_like,

        "description":
            (
                "Conservative reference plus lymphoid clusters; "
                "useful robustness check, but lymphoid clusters "
                "are DS3-biased."
            ),
    },

    "extended_myeloid": {
        "reference_clusters":
            core_reference
            + myeloid_reference,

        "query_clusters":
            query_tumor_like,

        "description":
            (
                "Extended reference including broadly represented "
                "myeloid/TAM clusters; excludes patient-dominant "
                "myeloid cluster 10."
            ),
    },

    "review_cycling": {
        "reference_clusters":
            core_reference,

        "query_clusters":
            review_cycling,

        "description":
            (
                "Review run for cycling cluster to decide whether "
                "cycling cells are tumor-like/malignant or "
                "normal/immune cycling."
            ),
    },

    "review_ambiguous": {
        "reference_clusters":
            core_reference,

        "query_clusters":
            review_ambiguous,

        "description":
            (
                "Review run for ambiguous mixed/MES-stromal-like "
                "clusters."
            ),
    },
}


strategy_rows = []

for name, info in strategies.items():
    strategy_rows.append(
        {
            "strategy":
                name,

            "reference_clusters":
                ",".join(
                    info[
                        "reference_clusters"
                    ]
                ),

            "query_clusters":
                ",".join(
                    info[
                        "query_clusters"
                    ]
                ),

            "description":
                info[
                    "description"
                ],

            "n_reference_clusters":
                len(
                    info[
                        "reference_clusters"
                    ]
                ),

            "n_query_clusters":
                len(
                    info[
                        "query_clusters"
                    ]
                ),
        }
    )


strategy_df = pd.DataFrame(
    strategy_rows
)

STRATEGY_TSV = (
    METADATA_DIR
    / "CNV_reference_strategy_definitions_HVG8000_r0_6.tsv"
)

write_tsv_replace(
    strategy_df,
    STRATEGY_TSV,
)

display(
    strategy_df
)


if strategy_df.shape != EXPECTED_STRATEGY_SHAPE:
    raise AssertionError(
        "Historical strategy-definition shape changed."
    )


cluster_role_counts = (
    obs
    .groupby(
        [
            CLUSTER_KEY,
            CNV_CLUSTER_ROLE_COL,
            ANNOTATION_COL,
            LINEAGE_COL,
            CONFIDENCE_COL,
        ],
        observed=True,
    )
    .size()
    .reset_index(
        name="n_cells"
    )
    .sort_values(
        CLUSTER_KEY,
        key=lambda series:
            series
            .astype(str)
            .map(
                lambda value:
                    int(value)
                    if value.isdigit()
                    else value
            ),
    )
)


if BIAS_STATUS_COL in obs.columns:
    bias_by_cluster = (
        obs[
            [
                CLUSTER_KEY,
                BIAS_STATUS_COL,
            ]
        ]
        .drop_duplicates()
        .copy()
    )

    cluster_role_counts = (
        cluster_role_counts.merge(
            bias_by_cluster,
            on=CLUSTER_KEY,
            how="left",
        )
    )


role_cell_counts = (
    obs[
        CNV_CLUSTER_ROLE_COL
    ]
    .astype(str)
    .value_counts()
    .rename_axis(
        "cnv_cluster_role"
    )
    .reset_index(
        name="n_cells"
    )
)

role_cell_counts[
    "fraction_cells"
] = (
    role_cell_counts[
        "n_cells"
    ]
    / EXPECTED_N_CELLS
)


CLUSTER_CATEGORY_TSV = (
    METADATA_DIR
    / "CNV_input_cluster_category_counts_HVG8000_r0_6.tsv"
)

CELL_CATEGORY_TSV = (
    METADATA_DIR
    / "CNV_input_cell_category_counts_HVG8000_r0_6.tsv"
)


write_tsv_replace(
    cluster_role_counts,
    CLUSTER_CATEGORY_TSV,
)

write_tsv_replace(
    role_cell_counts,
    CELL_CATEGORY_TSV,
)


if (
    cluster_role_counts.shape
    != EXPECTED_CLUSTER_CATEGORY_SHAPE
):
    raise AssertionError(
        "Historical cluster-category shape changed: "
        f"{cluster_role_counts.shape}"
    )

if (
    role_cell_counts.shape
    != EXPECTED_CELL_CATEGORY_SHAPE
):
    raise AssertionError(
        "Historical cell-category shape changed: "
        f"{role_cell_counts.shape}"
    )


observed_role_counts = dict(
    zip(
        role_cell_counts[
            "cnv_cluster_role"
        ].astype(str),
        role_cell_counts[
            "n_cells"
        ].astype(int),
    )
)


if observed_role_counts != EXPECTED_ROLE_CELL_COUNTS:
    raise RuntimeError(
        "CNV cluster-role cell counts differ from "
        "the archived executed notebook."
    )


print(
    "[HISTORICAL CNV ROLE COUNTS REPRODUCED]"
)

display(
    cluster_role_counts
)

display(
    role_cell_counts
)

[SAVED] G:\Repository\glioma-cnv-single-cell-atlas\metadata\CNV_reference_strategy_definitions_HVG8000_r0_6.tsv shape=(5, 6)


,strategy,reference_clusters,query_clusters,description,n_reference_clusters,n_query_clusters
0,core_conservative,"8,14","0,4,7,11",Most conservative CNV run: oligodendrocyte and...,2,4
1,core_plus_lymphoid,"8,14,5,15","0,4,7,11",Conservative reference plus lymphoid clusters;...,4,4
2,extended_myeloid,"8,14,1,3,6","0,4,7,11",Extended reference including broadly represent...,5,4
3,review_cycling,"8,14",2,Review run for cycling cluster to decide wheth...,2,1
4,review_ambiguous,"8,14","9,12",Review run for ambiguous mixed/MES-stromal-lik...,2,2


[SAVED] G:\Repository\glioma-cnv-single-cell-atlas\metadata\CNV_input_cluster_category_counts_HVG8000_r0_6.tsv shape=(18, 7)
[SAVED] G:\Repository\glioma-cnv-single-cell-atlas\metadata\CNV_input_cell_category_counts_HVG8000_r0_6.tsv shape=(9, 3)
[HISTORICAL CNV ROLE COUNTS REPRODUCED]


,leiden_scvi_main,cnv_cluster_role_HVG8000_r0_6,annotation_marker_based_HVG8000_r0_6,lineage_marker_based_HVG8000_r0_6,annotation_confidence_HVG8000_r0_6,n_cells,cluster_bias_status_HVG8000_r0_6
0,0,query_putative_tumor_like,putative_tumor_OPC_NPC_like,putative_tumor_like,moderate_high,61646,broadly_represented
1,1,reference_myeloid_extended_candidate,myeloid_C1Q_TREM2_APOE_TAM,myeloid,high,59791,broadly_represented
2,2,query_review_cycling,cycling_proliferating_cells,cycling_ambiguous,moderate,38634,review_ambiguous
3,3,reference_myeloid_extended_candidate,myeloid_inflammatory_MES_associated,myeloid,moderate,35355,broadly_represented
4,4,query_putative_tumor_like,putative_tumor_hypoxia_stress,putative_tumor_like_stress,moderate_high,32896,broadly_represented
5,5,reference_lymphoid_candidate,T_NK_cells,lymphoid,high,23977,dataset_dominant
6,6,reference_myeloid_extended_candidate,myeloid_APC_HLA_CD163_TAM,myeloid,high,22943,broadly_represented
7,7,query_putative_tumor_like,putative_tumor_AC_like_reactive,putative_tumor_like,moderate_high,22525,broadly_represented
8,8,reference_core_conservative,oligodendrocyte_myelinating,normal_brain_glial,high,14809,broadly_represented
9,9,query_review_ambiguous,ambiguous_mixed_Hb_T_NK_HLA,ambiguous_mixed_or_doublet,low,13864,review_ambiguous


,cnv_cluster_role,n_cells,fraction_cells
0,query_putative_tumor_like,123250,0.350713
1,reference_myeloid_extended_candidate,118089,0.336027
2,query_review_cycling,38634,0.109935
3,reference_lymphoid_candidate,25144,0.071548
4,query_review_ambiguous,18667,0.053118
5,reference_core_conservative,17449,0.049652
6,reference_myeloid_review_not_core,6340,0.018041
7,reference_stromal_review_not_core,3566,0.010147
8,exclude_or_low_confidence,288,0.000820


In [4]:
# =========================
# 3) GENCODE v49 GENE POSITIONS / CHROMOSOME ORDER
# =========================

if not GTF_GZ.exists():
    raise FileNotFoundError(
        GTF_GZ
    )


gtf_pos = parse_gtf_gene_positions(
    GTF_GZ
)

print(
    "GTF gene symbols:",
    gtf_pos.shape,
)


if gtf_pos.shape != (
    EXPECTED_GTF_UNIQUE_SYMBOLS,
    9,
):
    raise AssertionError(
        "GENCODE-v49 parsed symbol table changed: "
        f"{gtf_pos.shape}"
    )


var_names = pd.Index(
    var.index.astype(str)
)


var_df = pd.DataFrame(
    {
        "gene_symbol":
            var_names,
    }
)


matched = var_df.merge(
    gtf_pos,
    on="gene_symbol",
    how="left",
)

matched[
    "is_matched"
] = (
    matched[
        "chromosome"
    ].notna()
)


canonical_chroms = [
    str(index)
    for index
    in range(
        1,
        23,
    )
] + [
    "X",
    "Y",
]


matched[
    "is_canonical"
] = (
    matched[
        "chromosome"
    ]
    .astype(str)
    .isin(
        canonical_chroms
    )
)


matched[
    "usable_for_cnv_gene_order"
] = (
    matched[
        "is_matched"
    ]
    & matched[
        "is_canonical"
    ]
)


chrom_order = {
    str(index):
        index
    for index
    in range(
        1,
        23,
    )
}

chrom_order.update(
    {
        "X": 23,
        "Y": 24,
    }
)


matched[
    "chrom_order"
] = (
    matched[
        "chromosome"
    ]
    .map(
        chrom_order
    )
    .fillna(
        999
    )
    .astype(
        int
    )
)


matched = matched.sort_values(
    [
        "usable_for_cnv_gene_order",
        "chrom_order",
        "start",
        "gene_symbol",
    ],
    ascending=[
        False,
        True,
        True,
        True,
    ],
).copy()


usable = (
    matched[
        matched[
            "usable_for_cnv_gene_order"
        ]
    ]
    .copy()
)


usable_genes = (
    usable[
        "gene_symbol"
    ]
    .astype(str)
    .tolist()
)


position_report = pd.DataFrame(
    [
        {
            "n_genes_total":
                EXPECTED_N_GENES,

            "n_genes_matched_to_gtf":
                int(
                    matched[
                        "is_matched"
                    ].sum()
                ),

            "n_genes_canonical_usable":
                int(
                    matched[
                        "usable_for_cnv_gene_order"
                    ].sum()
                ),

            "fraction_matched_to_gtf":
                float(
                    matched[
                        "is_matched"
                    ].mean()
                ),

            "fraction_canonical_usable":
                float(
                    matched[
                        "usable_for_cnv_gene_order"
                    ].mean()
                ),

            "gtf_file":
                str(
                    GTF_GZ
                ),
        }
    ]
)


if int(
    position_report.iloc[
        0
    ][
        "n_genes_matched_to_gtf"
    ]
) != EXPECTED_N_GENES:
    raise AssertionError(
        "Not all 15,176 historical genes map to GENCODE v49."
    )


if int(
    position_report.iloc[
        0
    ][
        "n_genes_canonical_usable"
    ]
) != EXPECTED_GENE_ORDERED_GENES:
    raise AssertionError(
        "Historical canonical gene-order count changed."
    )


infercnv_gene_order = (
    usable[
        [
            "gene_symbol",
            "chromosome",
            "start",
            "end",
        ]
    ]
    .copy()
)

infercnv_gene_order.columns = [
    "gene",
    "chr",
    "start",
    "end",
]


unmatched = (
    matched[
        ~matched[
            "usable_for_cnv_gene_order"
        ]
    ]
    .copy()
)


GENE_POSITION_REPORT_TSV = (
    METADATA_DIR
    / "CNV_gene_position_readiness_report_HVG8000_r0_6.tsv"
)

GENE_MATCH_TSV = (
    METADATA_DIR
    / "CNV_gene_order_gencode_v49_matched_header_HVG8000_r0_6.tsv"
)

GENE_ORDER_NOHEADER_TSV = (
    METADATA_DIR
    / "CNV_gene_order_gencode_v49_matched_noheader_HVG8000_r0_6.tsv"
)

GENE_UNMATCHED_TSV = (
    METADATA_DIR
    / "CNV_gene_order_unmatched_genes_HVG8000_r0_6.tsv"
)


write_tsv_replace(
    position_report,
    GENE_POSITION_REPORT_TSV,
)

write_tsv_replace(
    infercnv_gene_order,
    GENE_MATCH_TSV,
)


if GENE_ORDER_NOHEADER_TSV.exists():
    GENE_ORDER_NOHEADER_TSV.unlink()

infercnv_gene_order.to_csv(
    GENE_ORDER_NOHEADER_TSV,
    sep="\t",
    index=False,
    header=False,
)

print(
    f"[SAVED] {GENE_ORDER_NOHEADER_TSV} "
    f"shape={infercnv_gene_order.shape}"
)


unmatched_export = (
    unmatched[
        [
            "gene_symbol",
            "is_matched",
            "chromosome",
            "start",
            "end",
            "gene_type",
        ]
    ]
    .copy()
)

write_tsv_replace(
    unmatched_export,
    GENE_UNMATCHED_TSV,
)


if position_report.shape != EXPECTED_POSITION_REPORT_SHAPE:
    raise AssertionError(
        "Historical gene-position report shape changed."
    )

if infercnv_gene_order.shape != EXPECTED_GENE_ORDER_SHAPE:
    raise AssertionError(
        "Historical gene-order table shape changed."
    )

if unmatched_export.shape != EXPECTED_UNMATCHED_SHAPE:
    raise AssertionError(
        "Historical unmatched-gene table shape changed."
    )


usable_source_positions = (
    var_names.get_indexer(
        usable_genes
    )
)

if np.any(
    usable_source_positions
    < 0
):
    raise RuntimeError(
        "Could not map chromosome-ordered genes "
        "back to the full-gene var order."
    )


geneordered_var = (
    var.iloc[
        usable_source_positions
    ]
    .copy()
)

geneordered_var.index = pd.Index(
    usable_genes,
    dtype=str,
)


coord_map = (
    infercnv_gene_order
    .set_index(
        "gene"
    )
)


geneordered_var[
    "cnv_chr"
] = (
    coord_map.loc[
        geneordered_var.index,
        "chr",
    ]
    .astype(str)
    .to_numpy()
)

geneordered_var[
    "cnv_start"
] = (
    coord_map.loc[
        geneordered_var.index,
        "start",
    ]
    .astype(int)
    .to_numpy()
)

geneordered_var[
    "cnv_end"
] = (
    coord_map.loc[
        geneordered_var.index,
        "end",
    ]
    .astype(int)
    .to_numpy()
)


print(
    "\n[HISTORICAL GENCODE-v49 GENE ORDER REPRODUCED]"
)

display(
    position_report
)

display(
    infercnv_gene_order.head()
)

display(
    unmatched_export.head()
)

GTF gene symbols: (77271, 9)
[SAVED] G:\Repository\glioma-cnv-single-cell-atlas\metadata\CNV_gene_position_readiness_report_HVG8000_r0_6.tsv shape=(1, 6)
[SAVED] G:\Repository\glioma-cnv-single-cell-atlas\metadata\CNV_gene_order_gencode_v49_matched_header_HVG8000_r0_6.tsv shape=(15163, 4)
[SAVED] G:\Repository\glioma-cnv-single-cell-atlas\metadata\CNV_gene_order_gencode_v49_matched_noheader_HVG8000_r0_6.tsv shape=(15163, 4)
[SAVED] G:\Repository\glioma-cnv-single-cell-atlas\metadata\CNV_gene_order_unmatched_genes_HVG8000_r0_6.tsv shape=(13, 6)

[HISTORICAL GENCODE-v49 GENE ORDER REPRODUCED]


,n_genes_total,n_genes_matched_to_gtf,n_genes_canonical_usable,fraction_matched_to_gtf,fraction_canonical_usable,gtf_file
0,15176,15176,15163,1.0,0.999143,G:\Repository\glioma-cnv-single-cell-atlas\dat...


,gene,chr,start,end
6667,LINC00115,1,586945,827989
4239,FAM87B,1,817363,819842
4211,FAM41C,1,868052,877032
11108,SAMD11,1,923923,944575
8461,NOC2L,1,943527,960714


,gene_symbol,is_matched,chromosome,start,end,gene_type
7873,MT-ND1,True,M,3307,4262,protein_coding
7874,MT-ND2,True,M,4470,5511,protein_coding
7869,MT-CO1,True,M,5904,7445,protein_coding
7870,MT-CO2,True,M,7586,8269,protein_coding
7868,MT-ATP8,True,M,8366,8572,protein_coding


In [5]:
# =========================
# 4) PRECOMPUTE STRATEGY ANNOTATIONS + HISTORICAL SAMPLE SIZES
# =========================

strategy_runtime = {}
strategy_manifest_rows = []
ref_group_rows = []


for strategy_name, info in strategies.items():

    ref_clusters = [
        str(value)
        for value
        in info[
            "reference_clusters"
        ]
    ]

    query_clusters = [
        str(value)
        for value
        in info[
            "query_clusters"
        ]
    ]


    cluster_series = (
        obs[
            CLUSTER_KEY
        ]
        .astype(str)
    )

    ref_mask = (
        cluster_series.isin(
            ref_clusters
        )
    )

    query_mask = (
        cluster_series.isin(
            query_clusters
        )
    )

    keep_mask = (
        ref_mask
        | query_mask
    )


    cnv_group = pd.Series(
        index=obs.index,
        dtype="object",
    )

    cnv_group.loc[
        ref_mask
    ] = "reference"

    cnv_group.loc[
        query_mask
    ] = "query"


    annotation = pd.DataFrame(
        {
            "cell_id":
                obs.index.astype(str),

            "cnv_group":
                cnv_group.astype(str),

            "cluster":
                cluster_series,

            "annotation":
                obs[
                    ANNOTATION_COL
                ].astype(str),

            "lineage":
                obs[
                    LINEAGE_COL
                ].astype(str),

            "condition":
                (
                    obs[
                        "condition"
                    ].astype(str)
                    if "condition"
                    in obs.columns
                    else pd.Series(
                        "unknown",
                        index=obs.index,
                    )
                ),

            "core_dataset":
                (
                    obs[
                        "core_dataset"
                    ].astype(str)
                    if "core_dataset"
                    in obs.columns
                    else pd.Series(
                        "unknown",
                        index=obs.index,
                    )
                ),

            "patient_id":
                (
                    obs[
                        "patient_id"
                    ].astype(str)
                    if "patient_id"
                    in obs.columns
                    else pd.Series(
                        "unknown",
                        index=obs.index,
                    )
                ),

            "sample_id":
                (
                    obs[
                        "sample_id"
                    ].astype(str)
                    if "sample_id"
                    in obs.columns
                    else pd.Series(
                        "unknown",
                        index=obs.index,
                    )
                ),
        }
    )

    annotation = (
        annotation[
            keep_mask.to_numpy()
        ]
        .copy()
    )


    sampled_ref = sample_cells_by_cluster(
        obs,
        ref_clusters,
        CLUSTER_KEY,
        MAX_REF_PER_STRATEGY,
        MAX_PER_CLUSTER_REF,
        RANDOM_SEED,
    )

    sampled_query = sample_cells_by_cluster(
        obs,
        query_clusters,
        CLUSTER_KEY,
        MAX_QUERY_PER_STRATEGY,
        MAX_PER_CLUSTER_QUERY,
        RANDOM_SEED + 1,
    )

    sampled_cells = list(
        dict.fromkeys(
            sampled_ref
            + sampled_query
        )
    )


    annotation_path = (
        METADATA_DIR
        / (
            "CNV_infercnv_cell_annotations_"
            f"{strategy_name}_HVG8000_r0_6.tsv"
        )
    )

    reference_group_file = (
        METADATA_DIR
        / (
            "CNV_reference_group_names_"
            f"{strategy_name}_HVG8000_r0_6.tsv"
        )
    )

    sampled_h5ad = (
        CNV_DIR
        / (
            "GBM_core_4datasets_CNV_test_subset_"
            f"{strategy_name}_HVG8000_r0_6_countsX.h5ad"
        )
    )


    expected = (
        EXPECTED_STRATEGY_COUNTS[
            strategy_name
        ]
    )


    observed = {
        "annotation":
            int(
                annotation.shape[
                    0
                ]
            ),

        "reference_all":
            int(
                ref_mask.sum()
            ),

        "query_all":
            int(
                query_mask.sum()
            ),

        "sampled_reference":
            len(
                sampled_ref
            ),

        "sampled_query":
            len(
                sampled_query
            ),

        "sampled_total":
            len(
                sampled_cells
            ),
    }


    if observed != expected:
        raise RuntimeError(
            f"Historical strategy counts changed "
            f"for {strategy_name}: "
            f"{observed} vs {expected}"
        )


    sampled_positions = (
        obs.index.get_indexer(
            sampled_cells
        )
    )

    if np.any(
        sampled_positions
        < 0
    ):
        raise RuntimeError(
            f"Could not map sampled cells "
            f"for {strategy_name}."
        )


    strategy_runtime[
        strategy_name
    ] = {
        "ref_clusters":
            ref_clusters,

        "query_clusters":
            query_clusters,

        "ref_mask":
            ref_mask,

        "query_mask":
            query_mask,

        "annotation":
            annotation,

        "annotation_path":
            annotation_path,

        "reference_group_file":
            reference_group_file,

        "sampled_h5ad":
            sampled_h5ad,

        "sampled_ref":
            sampled_ref,

        "sampled_query":
            sampled_query,

        "sampled_cells":
            sampled_cells,

        "sampled_positions":
            sampled_positions.astype(
                np.int64,
                copy=False,
            ),
    }


    strategy_manifest_rows.append(
        {
            "strategy":
                strategy_name,

            "reference_clusters":
                ",".join(
                    ref_clusters
                ),

            "query_clusters":
                ",".join(
                    query_clusters
                ),

            "n_cells_annotation_file":
                observed[
                    "annotation"
                ],

            "n_reference_cells_all":
                observed[
                    "reference_all"
                ],

            "n_query_cells_all":
                observed[
                    "query_all"
                ],

            "n_sampled_reference_cells":
                observed[
                    "sampled_reference"
                ],

            "n_sampled_query_cells":
                observed[
                    "sampled_query"
                ],

            "n_sampled_total_cells":
                observed[
                    "sampled_total"
                ],

            "annotation_tsv":
                str(
                    annotation_path
                ),

            "reference_group_file":
                str(
                    reference_group_file
                ),

            "sampled_h5ad":
                str(
                    sampled_h5ad
                ),
        }
    )


    ref_group_rows.append(
        {
            "strategy":
                strategy_name,

            "reference_group_name":
                "reference",

            "file":
                str(
                    reference_group_file
                ),
        }
    )


strategy_manifest = pd.DataFrame(
    strategy_manifest_rows
)

ref_group_df = pd.DataFrame(
    ref_group_rows
)


if strategy_manifest.shape != EXPECTED_STRATEGY_MANIFEST_SHAPE:
    raise AssertionError(
        "Historical strategy-manifest shape changed."
    )

if ref_group_df.shape != EXPECTED_REF_GROUPS_SHAPE:
    raise AssertionError(
        "Historical reference-group summary shape changed."
    )


print(
    "[HISTORICAL STRATEGY / SAMPLE-SIZE CHECKPOINTS REPRODUCED]"
)

display(
    strategy_manifest
)

[HISTORICAL STRATEGY / SAMPLE-SIZE CHECKPOINTS REPRODUCED]


,strategy,reference_clusters,query_clusters,n_cells_annotation_file,n_reference_cells_all,n_query_cells_all,n_sampled_reference_cells,n_sampled_query_cells,n_sampled_total_cells,annotation_tsv,reference_group_file,sampled_h5ad
0,core_conservative,"8,14","0,4,7,11",140699,17449,123250,7640,25000,32640,G:\Repository\glioma-cnv-single-cell-atlas\met...,G:\Repository\glioma-cnv-single-cell-atlas\met...,G:\Repository\glioma-cnv-single-cell-atlas\dat...
1,core_plus_lymphoid,"8,14,5,15","0,4,7,11",165843,42593,123250,13807,25000,38807,G:\Repository\glioma-cnv-single-cell-atlas\met...,G:\Repository\glioma-cnv-single-cell-atlas\met...,G:\Repository\glioma-cnv-single-cell-atlas\dat...
2,extended_myeloid,"8,14,1,3,6","0,4,7,11",258788,135538,123250,20000,25000,45000,G:\Repository\glioma-cnv-single-cell-atlas\met...,G:\Repository\glioma-cnv-single-cell-atlas\met...,G:\Repository\glioma-cnv-single-cell-atlas\dat...
3,review_cycling,"8,14",2,56083,17449,38634,7640,7000,14640,G:\Repository\glioma-cnv-single-cell-atlas\met...,G:\Repository\glioma-cnv-single-cell-atlas\met...,G:\Repository\glioma-cnv-single-cell-atlas\dat...
4,review_ambiguous,"8,14","9,12",36116,17449,18667,7640,11803,19443,G:\Repository\glioma-cnv-single-cell-atlas\met...,G:\Repository\glioma-cnv-single-cell-atlas\met...,G:\Repository\glioma-cnv-single-cell-atlas\dat...


In [6]:
# =========================
# 5) RESUMABLE STREAMING BUILD:
#    ALL-CELL FULL-GENE + CHROMOSOME-ORDERED H5ADs
# =========================

def initialize_sparse_h5ad(
    path,
    obs_frame,
    var_frame,
    shape,
    data_dtype,
    indices_dtype,
    indptr_dtype,
    obsm_dict,
    uns_dict,
):
    path = Path(
        path
    )

    if path.exists():
        path.unlink()


    skeleton = ad.AnnData(
        X=None,
        obs=obs_frame.copy(),
        var=var_frame.copy(),
        obsm={
            key:
                value
            for key, value
            in obsm_dict.items()
            if value is not None
        },
        uns=uns_dict.copy(),
    )


    skeleton.write_h5ad(
        path,
        compression="gzip",
    )


    del skeleton
    gc.collect()


    with h5py.File(
        path,
        "r+",
    ) as handle:

        if "X" in handle:
            del handle[
                "X"
            ]

        x_group = handle.create_group(
            "X"
        )

        x_group.attrs[
            "encoding-type"
        ] = "csr_matrix"

        x_group.attrs[
            "encoding-version"
        ] = "0.1.0"

        x_group.attrs[
            "shape"
        ] = np.asarray(
            shape,
            dtype=np.int64,
        )


        x_group.create_dataset(
            "data",
            shape=(
                0,
            ),
            maxshape=(
                None,
            ),
            dtype=np.dtype(
                data_dtype
            ),
            chunks=(
                262_144,
            ),
            compression="gzip",
            compression_opts=4,
        )


        x_group.create_dataset(
            "indices",
            shape=(
                0,
            ),
            maxshape=(
                None,
            ),
            dtype=np.dtype(
                indices_dtype
            ),
            chunks=(
                262_144,
            ),
            compression="gzip",
            compression_opts=4,
        )


        x_group.create_dataset(
            "indptr",
            shape=(
                shape[
                    0
                ]
                + 1,
            ),
            dtype=np.dtype(
                indptr_dtype
            ),
            chunks=(
                min(
                    shape[
                        0
                    ]
                    + 1,
                    262_144,
                ),
            ),
            compression="gzip",
            compression_opts=4,
        )


        x_group[
            "indptr"
        ][
            0
        ] = 0

        handle.flush()


def append_csr_rows(
    path,
    matrix,
    row_start,
    row_end,
    committed_nnz,
):
    with h5py.File(
        path,
        "r+",
    ) as handle:

        x_group = handle[
            "X"
        ]

        data_ds = x_group[
            "data"
        ]

        indices_ds = x_group[
            "indices"
        ]

        indptr_ds = x_group[
            "indptr"
        ]


        if int(
            data_ds.shape[
                0
            ]
        ) != int(
            committed_nnz
        ):
            raise RuntimeError(
                f"Unexpected committed nnz in {path}."
            )


        new_nnz = (
            int(
                committed_nnz
            )
            + int(
                matrix.nnz
            )
        )


        data_ds.resize(
            (
                new_nnz,
            )
        )

        indices_ds.resize(
            (
                new_nnz,
            )
        )


        if matrix.nnz:
            data_ds[
                committed_nnz:new_nnz
            ] = np.asarray(
                matrix.data,
                dtype=data_ds.dtype,
            )

            indices_ds[
                committed_nnz:new_nnz
            ] = np.asarray(
                matrix.indices,
                dtype=indices_ds.dtype,
            )


        global_indptr = (
            np.asarray(
                matrix.indptr,
                dtype=np.int64,
            )
            + int(
                committed_nnz
            )
        )


        indptr_ds[
            row_start:row_end + 1
        ] = global_indptr.astype(
            indptr_ds.dtype,
            copy=False,
        )


        handle.flush()


    return new_nnz


def truncate_to_checkpoint(
    path,
    committed_nnz,
):
    with h5py.File(
        path,
        "r+",
    ) as handle:

        handle[
            "X"
        ][
            "data"
        ].resize(
            (
                int(
                    committed_nnz
                ),
            )
        )

        handle[
            "X"
        ][
            "indices"
        ].resize(
            (
                int(
                    committed_nnz
                ),
            )
        )

        handle.flush()


source_fingerprint = {
    "path":
        str(
            input_path.resolve()
        ),

    "size":
        int(
            input_path.stat().st_size
        ),

    "mtime_ns":
        int(
            input_path.stat().st_mtime_ns
        ),

    "counts_shape":
        list(
            counts_shape
        ),

    "counts_nnz":
        counts_nnz,

    "obs_names_sha256":
        hash_strings(
            obs.index.astype(str)
        ),

    "var_names_sha256":
        hash_strings(
            var_names
        ),

    "usable_genes_sha256":
        hash_strings(
            usable_genes
        ),
}


progress = None

if STREAM_PROGRESS_JSON.exists():
    try:
        with STREAM_PROGRESS_JSON.open(
            "r",
            encoding="utf-8",
        ) as handle:
            candidate = json.load(
                handle
            )

        if (
            candidate.get(
                "source_fingerprint"
            )
            == source_fingerprint
        ):
            progress = candidate

    except Exception:
        progress = None


if (
    progress is None
    or not SCRATCH_ALLCELLS_H5AD.exists()
    or not SCRATCH_GENE_ORDERED_H5AD.exists()
):
    print(
        "Initializing fresh streaming CNV-input build."
    )


    for path in (
        SCRATCH_ALLCELLS_H5AD,
        SCRATCH_GENE_ORDERED_H5AD,
        STREAM_PROGRESS_JSON,
    ):
        if Path(
            path
        ).exists():
            Path(
                path
            ).unlink()


    full_obsm = {
        "X_scVI":
            x_scvi,

        "X_umap":
            x_umap,
    }


    allcells_uns = {
        "CNV_input_note":
            (
                "Raw counts are stored in adata.X. "
                "The duplicate counts layer is intentionally absent "
                "from this CNV export."
            ),

        "CNV_cluster_key":
            CLUSTER_KEY,

        "CNV_main_query_clusters":
            ",".join(
                query_tumor_like
            ),

        "CNV_core_reference_clusters":
            ",".join(
                core_reference
            ),

        "memory_safe_build":
            True,
    }


    geneordered_uns = dict(
        allcells_uns
    )

    geneordered_uns[
        "CNV_gene_order_file_header"
    ] = str(
        GENE_MATCH_TSV
    )

    geneordered_uns[
        "CNV_gene_order_file_noheader"
    ] = str(
        GENE_ORDER_NOHEADER_TSV
    )


    initialize_sparse_h5ad(
        SCRATCH_ALLCELLS_H5AD,
        obs,
        var,
        (
            EXPECTED_N_CELLS,
            EXPECTED_N_GENES,
        ),
        counts_data_dtype,
        counts_indices_dtype,
        counts_indptr_dtype,
        full_obsm,
        allcells_uns,
    )


    initialize_sparse_h5ad(
        SCRATCH_GENE_ORDERED_H5AD,
        obs,
        geneordered_var,
        (
            EXPECTED_N_CELLS,
            EXPECTED_GENE_ORDERED_GENES,
        ),
        counts_data_dtype,
        counts_indices_dtype,
        counts_indptr_dtype,
        full_obsm,
        geneordered_uns,
    )


    progress = {
        "source_fingerprint":
            source_fingerprint,

        "next_row":
            0,

        "allcells_nnz":
            0,

        "geneordered_nnz":
            0,

        "complete":
            False,
    }


    write_json_atomic(
        progress,
        STREAM_PROGRESS_JSON,
    )


# Roll back any partially appended uncommitted block.
truncate_to_checkpoint(
    SCRATCH_ALLCELLS_H5AD,
    progress[
        "allcells_nnz"
    ],
)

truncate_to_checkpoint(
    SCRATCH_GENE_ORDERED_H5AD,
    progress[
        "geneordered_nnz"
    ],
)


if progress[
    "complete"
]:
    print(
        "[REUSING] completed streamed CNV-input H5ADs."
    )

else:
    r0 = int(
        progress[
            "next_row"
        ]
    )

    print(
        "Resuming streamed build at row:",
        r0,
    )


    with h5py.File(
        input_path,
        "r",
    ) as source_handle:

        source_counts = (
            source_handle[
                "layers"
            ][
                "counts"
            ]
        )

        data_ds = source_counts[
            "data"
        ]

        indices_ds = source_counts[
            "indices"
        ]

        indptr = np.asarray(
            source_counts[
                "indptr"
            ],
            dtype=np.int64,
        )


        while r0 < EXPECTED_N_CELLS:

            d0 = int(
                indptr[
                    r0
                ]
            )

            target_d1 = min(
                d0
                + MAX_NNZ_PER_ROW_BLOCK,
                counts_nnz,
            )

            r1 = int(
                np.searchsorted(
                    indptr,
                    target_d1,
                    side="right",
                )
                - 1
            )

            r1 = max(
                r1,
                r0 + 1,
            )

            r1 = min(
                r1,
                EXPECTED_N_CELLS,
            )

            d1 = int(
                indptr[
                    r1
                ]
            )


            block_data = np.array(
                data_ds[
                    d0:d1
                ],
                copy=True,
            )

            block_indices = np.array(
                indices_ds[
                    d0:d1
                ],
                copy=True,
            )

            block_indptr = (
                indptr[
                    r0:r1 + 1
                ]
                - d0
            ).astype(
                np.int64,
                copy=True,
            )


            block = sp.csr_matrix(
                (
                    block_data,
                    block_indices,
                    block_indptr,
                ),
                shape=(
                    r1 - r0,
                    EXPECTED_N_GENES,
                ),
                copy=False,
            )

            block.sort_indices()


            geneordered_block = (
                block[
                    :,
                    usable_source_positions,
                ]
                .tocsr()
            )

            geneordered_block.sort_indices()


            allcells_nnz = append_csr_rows(
                SCRATCH_ALLCELLS_H5AD,
                block,
                r0,
                r1,
                progress[
                    "allcells_nnz"
                ],
            )


            geneordered_nnz = append_csr_rows(
                SCRATCH_GENE_ORDERED_H5AD,
                geneordered_block,
                r0,
                r1,
                progress[
                    "geneordered_nnz"
                ],
            )


            progress[
                "next_row"
            ] = int(
                r1
            )

            progress[
                "allcells_nnz"
            ] = int(
                allcells_nnz
            )

            progress[
                "geneordered_nnz"
            ] = int(
                geneordered_nnz
            )


            write_json_atomic(
                progress,
                STREAM_PROGRESS_JSON,
            )


            print(
                "[STREAMED]",
                f"rows {r0:,}-{r1:,}",
                f"full nnz={allcells_nnz:,}",
                f"gene-ordered nnz={geneordered_nnz:,}",
            )


            del block
            del geneordered_block
            del block_data
            del block_indices
            del block_indptr

            gc.collect()

            r0 = r1


    progress[
        "complete"
    ] = True

    write_json_atomic(
        progress,
        STREAM_PROGRESS_JSON,
    )


print(
    "\n[STREAMING BUILD COMPLETE]"
)

print(
    "allcells scratch:",
    SCRATCH_ALLCELLS_H5AD,
)

print(
    "gene-ordered scratch:",
    SCRATCH_GENE_ORDERED_H5AD,
)

Initializing fresh streaming CNV-input build.
Resuming streamed build at row: 0
[STREAMED] rows 0-4,241 full nnz=7,998,673 gene-ordered nnz=7,973,102
[STREAMED] rows 4,241-11,672 full nnz=15,998,368 gene-ordered nnz=15,938,200
[STREAMED] rows 11,672-21,842 full nnz=23,998,353 gene-ordered nnz=23,921,684
[STREAMED] rows 21,842-31,237 full nnz=31,998,154 gene-ordered nnz=31,909,298
[STREAMED] rows 31,237-39,049 full nnz=39,996,920 gene-ordered nnz=39,901,257
[STREAMED] rows 39,049-45,782 full nnz=47,995,616 gene-ordered nnz=47,881,392
[STREAMED] rows 45,782-49,725 full nnz=55,994,798 gene-ordered nnz=55,841,134
[STREAMED] rows 49,725-54,177 full nnz=63,992,114 gene-ordered nnz=63,791,497
[STREAMED] rows 54,177-58,699 full nnz=71,991,266 gene-ordered nnz=71,742,942
[STREAMED] rows 58,699-63,156 full nnz=79,990,512 gene-ordered nnz=79,696,781
[STREAMED] rows 63,156-67,223 full nnz=87,990,258 gene-ordered nnz=87,654,125
[STREAMED] rows 67,223-70,841 full nnz=95,987,949 gene-ordered nnz=95,6

In [7]:
# =========================
# 6) VALIDATE + PUBLISH ALL-CELL CNV INPUTS
# =========================

def validate_cnv_h5ad(
    path,
    expected_shape,
    expected_var_names=None,
):
    path = Path(
        path
    )

    if not path.exists():
        return False


    check = sc.read_h5ad(
        path,
        backed="r",
    )

    try:
        if tuple(
            check.shape
        ) != tuple(
            expected_shape
        ):
            return False

        if CNV_CLUSTER_ROLE_COL not in check.obs.columns:
            return False

        if "counts" in check.layers:
            return False

        if expected_var_names is not None:
            if not check.var_names.equals(
                pd.Index(
                    expected_var_names
                )
            ):
                return False

        return True

    finally:
        check.file.close()


if not validate_cnv_h5ad(
    SCRATCH_ALLCELLS_H5AD,
    (
        EXPECTED_N_CELLS,
        EXPECTED_N_GENES,
    ),
    var_names,
):
    raise RuntimeError(
        "Scratch all-cell full-gene H5AD failed validation."
    )


if not validate_cnv_h5ad(
    SCRATCH_GENE_ORDERED_H5AD,
    (
        EXPECTED_N_CELLS,
        EXPECTED_GENE_ORDERED_GENES,
    ),
    usable_genes,
):
    raise RuntimeError(
        "Scratch gene-ordered H5AD failed validation."
    )


def publish_from_scratch(
    scratch_path,
    destination,
):
    scratch_path = Path(
        scratch_path
    )

    destination = Path(
        destination
    )

    destination.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    publishing = (
        destination.with_suffix(
            ".publishing.h5ad"
        )
    )

    if publishing.exists():
        publishing.unlink()


    required_space = (
        scratch_path.stat().st_size
        + 1024**3
    )

    free_space = shutil.disk_usage(
        destination.parent
    ).free

    if free_space < required_space:
        raise OSError(
            f"Not enough free space to publish {destination.name}."
        )


    print(
        "[PUBLISH]",
        scratch_path,
        "->",
        destination,
    )


    shutil.copyfile(
        scratch_path,
        publishing,
    )


    if publishing.stat().st_size != scratch_path.stat().st_size:
        raise IOError(
            "Published temporary file size mismatch."
        )


    os.replace(
        publishing,
        destination,
    )


    print(
        "[PUBLISHED]",
        destination,
    )


publish_from_scratch(
    SCRATCH_ALLCELLS_H5AD,
    ALLCELLS_H5AD,
)

publish_from_scratch(
    SCRATCH_GENE_ORDERED_H5AD,
    GENE_ORDERED_H5AD,
)


if not validate_cnv_h5ad(
    ALLCELLS_H5AD,
    (
        EXPECTED_N_CELLS,
        EXPECTED_N_GENES,
    ),
    var_names,
):
    raise RuntimeError(
        "Published all-cell full-gene H5AD failed validation."
    )


if not validate_cnv_h5ad(
    GENE_ORDERED_H5AD,
    (
        EXPECTED_N_CELLS,
        EXPECTED_GENE_ORDERED_GENES,
    ),
    usable_genes,
):
    raise RuntimeError(
        "Published gene-ordered H5AD failed validation."
    )


print(
    "\nAll-cell CNV exports complete"
)

print(
    "allcells:",
    ALLCELLS_H5AD,
)

print(
    "gene_ordered:",
    GENE_ORDERED_H5AD,
)

[PUBLISH] E:\glioma-cnv-scratch\main_analysis_10\GBM_core_4datasets_CNV_input_HVG8000_r0_6_allcells_fullgene_countsX.h5ad -> G:\Repository\glioma-cnv-single-cell-atlas\data\interim\integrated\CNV_inputs\GBM_core_4datasets_CNV_input_HVG8000_r0_6_allcells_fullgene_countsX.h5ad
[PUBLISHED] G:\Repository\glioma-cnv-single-cell-atlas\data\interim\integrated\CNV_inputs\GBM_core_4datasets_CNV_input_HVG8000_r0_6_allcells_fullgene_countsX.h5ad
[PUBLISH] E:\glioma-cnv-scratch\main_analysis_10\GBM_core_4datasets_CNV_input_HVG8000_r0_6_allcells_gene_ordered_countsX.h5ad -> G:\Repository\glioma-cnv-single-cell-atlas\data\interim\integrated\CNV_inputs\GBM_core_4datasets_CNV_input_HVG8000_r0_6_allcells_gene_ordered_countsX.h5ad
[PUBLISHED] G:\Repository\glioma-cnv-single-cell-atlas\data\interim\integrated\CNV_inputs\GBM_core_4datasets_CNV_input_HVG8000_r0_6_allcells_gene_ordered_countsX.h5ad

All-cell CNV exports complete
allcells: G:\Repository\glioma-cnv-single-cell-atlas\data\interim\integrated\CN

In [8]:
# =========================
# 7) EXPORT STRATEGY ANNOTATIONS + MEMORY-SAFE SAMPLED H5ADs
# =========================

def read_selected_rows_sequential(
    h5ad_path,
    target_positions,
):
    """
    Read a sampled row set from a CSR H5AD with sequential full-file access.
    Returned rows are restored to the exact requested target order.
    """
    target_positions = np.asarray(
        target_positions,
        dtype=np.int64,
    )


    if (
        target_positions.ndim != 1
        or np.any(
            target_positions < 0
        )
        or np.any(
            target_positions
            >= EXPECTED_N_CELLS
        )
    ):
        raise ValueError(
            "Invalid sampled row positions."
        )


    if len(
        np.unique(
            target_positions
        )
    ) != len(
        target_positions
    ):
        raise ValueError(
            "Sampled row positions are not unique."
        )


    wanted = np.zeros(
        EXPECTED_N_CELLS,
        dtype=bool,
    )

    wanted[
        target_positions
    ] = True


    matrix_parts = []
    global_row_parts = []


    with h5py.File(
        h5ad_path,
        "r",
    ) as handle:

        x_group = handle[
            "X"
        ]

        indptr = np.asarray(
            x_group[
                "indptr"
            ],
            dtype=np.int64,
        )

        data_ds = x_group[
            "data"
        ]

        indices_ds = x_group[
            "indices"
        ]

        total_nnz = int(
            data_ds.shape[
                0
            ]
        )


        r0 = 0

        while r0 < EXPECTED_N_CELLS:

            d0 = int(
                indptr[
                    r0
                ]
            )

            target_d1 = min(
                d0
                + MAX_NNZ_PER_ROW_BLOCK,
                total_nnz,
            )

            r1 = int(
                np.searchsorted(
                    indptr,
                    target_d1,
                    side="right",
                )
                - 1
            )

            r1 = max(
                r1,
                r0 + 1,
            )

            r1 = min(
                r1,
                EXPECTED_N_CELLS,
            )

            local_selected = np.flatnonzero(
                wanted[
                    r0:r1
                ]
            )


            if local_selected.size:

                d1 = int(
                    indptr[
                        r1
                    ]
                )

                block_data = np.array(
                    data_ds[
                        d0:d1
                    ],
                    copy=True,
                )

                block_indices = np.array(
                    indices_ds[
                        d0:d1
                    ],
                    copy=True,
                )

                block_indptr = (
                    indptr[
                        r0:r1 + 1
                    ]
                    - d0
                ).astype(
                    np.int64,
                    copy=True,
                )


                block = sp.csr_matrix(
                    (
                        block_data,
                        block_indices,
                        block_indptr,
                    ),
                    shape=(
                        r1 - r0,
                        EXPECTED_GENE_ORDERED_GENES,
                    ),
                    copy=False,
                )


                selected_block = (
                    block[
                        local_selected,
                        :,
                    ]
                    .tocsr()
                )


                matrix_parts.append(
                    selected_block
                )

                global_row_parts.append(
                    (
                        local_selected
                        + r0
                    ).astype(
                        np.int64,
                        copy=False,
                    )
                )


                del block
                del block_data
                del block_indices
                del block_indptr

                gc.collect()


            r0 = r1


    if not matrix_parts:
        raise RuntimeError(
            "No sampled rows were extracted."
        )


    matrix_sorted = sp.vstack(
        matrix_parts,
        format="csr",
    )

    global_rows_sorted = np.concatenate(
        global_row_parts
    )


    if matrix_sorted.shape[
        0
    ] != len(
        target_positions
    ):
        raise RuntimeError(
            "Sampled row extraction count mismatch."
        )


    lookup = np.full(
        EXPECTED_N_CELLS,
        -1,
        dtype=np.int64,
    )

    lookup[
        global_rows_sorted
    ] = np.arange(
        len(
            global_rows_sorted
        ),
        dtype=np.int64,
    )


    reorder = lookup[
        target_positions
    ]


    if np.any(
        reorder < 0
    ):
        raise RuntimeError(
            "Could not restore requested sampled row order."
        )


    matrix_target = (
        matrix_sorted[
            reorder,
            :,
        ]
        .tocsr()
    )


    del matrix_sorted
    del matrix_parts
    del global_row_parts
    gc.collect()


    return matrix_target


def sampled_output_valid(
    path,
    expected_cells,
    expected_obs_names,
):
    path = Path(
        path
    )

    if not path.exists():
        return False


    check = sc.read_h5ad(
        path,
        backed="r",
    )

    try:
        if check.shape != (
            expected_cells,
            EXPECTED_GENE_ORDERED_GENES,
        ):
            return False

        if not check.obs_names.equals(
            pd.Index(
                expected_obs_names
            )
        ):
            return False

        if (
            "cnv_sampled_strategy"
            not in check.obs.columns
            or "cnv_group"
            not in check.obs.columns
        ):
            return False

        return True

    finally:
        check.file.close()


for strategy_name, runtime in strategy_runtime.items():

    annotation = runtime[
        "annotation"
    ]

    annotation_path = runtime[
        "annotation_path"
    ]

    reference_group_file = runtime[
        "reference_group_file"
    ]

    sampled_h5ad = runtime[
        "sampled_h5ad"
    ]

    sampled_cells = runtime[
        "sampled_cells"
    ]

    sampled_positions = runtime[
        "sampled_positions"
    ]

    ref_clusters = runtime[
        "ref_clusters"
    ]


    write_tsv_replace(
        annotation,
        annotation_path,
    )

    write_text_replace(
        [
            "reference"
        ],
        reference_group_file,
    )


    if sampled_output_valid(
        sampled_h5ad,
        len(
            sampled_cells
        ),
        sampled_cells,
    ):
        print(
            "[REUSING] validated sampled H5AD:",
            sampled_h5ad,
        )

        continue


    print(
        "\n[BUILD SAMPLED H5AD]",
        strategy_name,
        len(
            sampled_cells
        ),
        "cells",
    )


    sampled_matrix = read_selected_rows_sequential(
        SCRATCH_GENE_ORDERED_H5AD,
        sampled_positions,
    )


    sampled_obs = (
        obs.iloc[
            sampled_positions
        ]
        .copy()
    )

    sampled_obs.index = pd.Index(
        sampled_cells,
        dtype=str,
    )

    sampled_obs[
        "cnv_sampled_strategy"
    ] = strategy_name

    sampled_obs[
        "cnv_group"
    ] = np.where(
        sampled_obs[
            CLUSTER_KEY
        ]
        .astype(str)
        .isin(
            ref_clusters
        ),
        "reference",
        "query",
    )


    sampled_obsm = {}

    if x_scvi is not None:
        sampled_obsm[
            "X_scVI"
        ] = x_scvi[
            sampled_positions
        ]

    if x_umap is not None:
        sampled_obsm[
            "X_umap"
        ] = x_umap[
            sampled_positions
        ]


    subset = ad.AnnData(
        X=sampled_matrix,
        obs=sampled_obs,
        var=geneordered_var.copy(),
        obsm=sampled_obsm,
        uns={
            "CNV_input_note":
                (
                    "Raw counts are stored in X. "
                    "This is a deterministic strategy-specific "
                    "sample from the canonical repository object."
                ),

            "CNV_cluster_key":
                CLUSTER_KEY,

            "CNV_strategy":
                strategy_name,

            "CNV_gene_order_file_header":
                str(
                    GENE_MATCH_TSV
                ),

            "CNV_gene_order_file_noheader":
                str(
                    GENE_ORDER_NOHEADER_TSV
                ),
        },
    )


    scratch_subset = (
        SCRATCH_ROOT
        / sampled_h5ad.name
    )

    if scratch_subset.exists():
        scratch_subset.unlink()


    subset.write_h5ad(
        scratch_subset,
        compression="gzip",
    )


    del subset
    del sampled_matrix
    gc.collect()


    publish_from_scratch(
        scratch_subset,
        sampled_h5ad,
    )


    if not sampled_output_valid(
        sampled_h5ad,
        len(
            sampled_cells
        ),
        sampled_cells,
    ):
        raise RuntimeError(
            f"Published sampled H5AD failed validation: "
            f"{sampled_h5ad}"
        )


STRATEGY_MANIFEST_TSV = (
    METADATA_DIR
    / "CNV_strategy_input_manifest_HVG8000_r0_6.tsv"
)

REF_GROUPS_TSV = (
    METADATA_DIR
    / "CNV_reference_group_names_HVG8000_r0_6.tsv"
)


write_tsv_replace(
    strategy_manifest,
    STRATEGY_MANIFEST_TSV,
)

write_tsv_replace(
    ref_group_df,
    REF_GROUPS_TSV,
)


print(
    "\n[ALL STRATEGY INPUTS PREPARED]"
)

display(
    strategy_manifest
)

[SAVED] G:\Repository\glioma-cnv-single-cell-atlas\metadata\CNV_infercnv_cell_annotations_core_conservative_HVG8000_r0_6.tsv shape=(140699, 9)
[SAVED] G:\Repository\glioma-cnv-single-cell-atlas\metadata\CNV_reference_group_names_core_conservative_HVG8000_r0_6.tsv n_lines=1

[BUILD SAMPLED H5AD] core_conservative 32640 cells
[PUBLISH] E:\glioma-cnv-scratch\main_analysis_10\GBM_core_4datasets_CNV_test_subset_core_conservative_HVG8000_r0_6_countsX.h5ad -> G:\Repository\glioma-cnv-single-cell-atlas\data\interim\integrated\CNV_inputs\GBM_core_4datasets_CNV_test_subset_core_conservative_HVG8000_r0_6_countsX.h5ad
[PUBLISHED] G:\Repository\glioma-cnv-single-cell-atlas\data\interim\integrated\CNV_inputs\GBM_core_4datasets_CNV_test_subset_core_conservative_HVG8000_r0_6_countsX.h5ad
[SAVED] G:\Repository\glioma-cnv-single-cell-atlas\metadata\CNV_infercnv_cell_annotations_core_plus_lymphoid_HVG8000_r0_6.tsv shape=(165843, 9)
[SAVED] G:\Repository\glioma-cnv-single-cell-atlas\metadata\CNV_reference

,strategy,reference_clusters,query_clusters,n_cells_annotation_file,n_reference_cells_all,n_query_cells_all,n_sampled_reference_cells,n_sampled_query_cells,n_sampled_total_cells,annotation_tsv,reference_group_file,sampled_h5ad
0,core_conservative,"8,14","0,4,7,11",140699,17449,123250,7640,25000,32640,G:\Repository\glioma-cnv-single-cell-atlas\met...,G:\Repository\glioma-cnv-single-cell-atlas\met...,G:\Repository\glioma-cnv-single-cell-atlas\dat...
1,core_plus_lymphoid,"8,14,5,15","0,4,7,11",165843,42593,123250,13807,25000,38807,G:\Repository\glioma-cnv-single-cell-atlas\met...,G:\Repository\glioma-cnv-single-cell-atlas\met...,G:\Repository\glioma-cnv-single-cell-atlas\dat...
2,extended_myeloid,"8,14,1,3,6","0,4,7,11",258788,135538,123250,20000,25000,45000,G:\Repository\glioma-cnv-single-cell-atlas\met...,G:\Repository\glioma-cnv-single-cell-atlas\met...,G:\Repository\glioma-cnv-single-cell-atlas\dat...
3,review_cycling,"8,14",2,56083,17449,38634,7640,7000,14640,G:\Repository\glioma-cnv-single-cell-atlas\met...,G:\Repository\glioma-cnv-single-cell-atlas\met...,G:\Repository\glioma-cnv-single-cell-atlas\dat...
4,review_ambiguous,"8,14","9,12",36116,17449,18667,7640,11803,19443,G:\Repository\glioma-cnv-single-cell-atlas\met...,G:\Repository\glioma-cnv-single-cell-atlas\met...,G:\Repository\glioma-cnv-single-cell-atlas\dat...


In [9]:
# =========================
# 8) STRATEGY COMPOSITION SUMMARIES
# =========================

composition_rows = []


for strategy_name, info in strategies.items():

    ref_clusters = [
        str(value)
        for value
        in info[
            "reference_clusters"
        ]
    ]

    query_clusters = [
        str(value)
        for value
        in info[
            "query_clusters"
        ]
    ]


    cluster_series = (
        obs[
            CLUSTER_KEY
        ]
        .astype(str)
    )


    group = pd.Series(
        "other",
        index=obs.index,
    )

    group.loc[
        cluster_series.isin(
            ref_clusters
        )
    ] = "reference"

    group.loc[
        cluster_series.isin(
            query_clusters
        )
    ] = "query"


    keep = group.isin(
        [
            "reference",
            "query",
        ]
    )


    tmp = (
        obs.loc[
            keep,
            [
                CLUSTER_KEY,
                ANNOTATION_COL,
                LINEAGE_COL,
            ],
        ]
        .copy()
    )

    tmp[
        "strategy"
    ] = strategy_name

    tmp[
        "cnv_group"
    ] = group.loc[
        keep
    ].to_numpy()


    for column in (
        "core_dataset",
        "condition",
        "patient_id",
        "sample_id",
    ):
        tmp[
            column
        ] = (
            obs.loc[
                keep,
                column,
            ]
            .astype(str)
            .to_numpy()
            if column
            in obs.columns
            else "unknown"
        )


    for group_name in (
        "reference",
        "query",
    ):
        sub = tmp[
            tmp[
                "cnv_group"
            ]
            == group_name
        ]

        if sub.empty:
            continue


        row = {
            "strategy":
                strategy_name,

            "cnv_group":
                group_name,

            "n_cells":
                sub.shape[
                    0
                ],

            "n_clusters":
                sub[
                    CLUSTER_KEY
                ].nunique(),
        }


        for column in (
            "core_dataset",
            "condition",
            "patient_id",
            "sample_id",
        ):
            counts = sub[
                column
            ].value_counts(
                normalize=False
            )

            row[
                f"top_{column}"
            ] = (
                counts.index[
                    0
                ]
            )

            row[
                f"top_{column}_n"
            ] = int(
                counts.iloc[
                    0
                ]
            )

            row[
                f"top_{column}_fraction"
            ] = float(
                counts.iloc[
                    0
                ]
                / sub.shape[
                    0
                ]
            )

            row[
                f"n_{column}"
            ] = int(
                counts.shape[
                    0
                ]
            )


        composition_rows.append(
            row
        )


composition_summary = pd.DataFrame(
    composition_rows
)


COMPOSITION_TSV = (
    METADATA_DIR
    / "CNV_strategy_composition_summary_HVG8000_r0_6.tsv"
)


write_tsv_replace(
    composition_summary,
    COMPOSITION_TSV,
)


if composition_summary.shape != EXPECTED_COMPOSITION_SHAPE:
    raise AssertionError(
        "Historical strategy-composition shape changed."
    )


# Hard-check strategy/group cell counts and cluster counts.
expected_composition_core = {
    ("core_conservative", "reference"):
        (17_449, 2),

    ("core_conservative", "query"):
        (123_250, 4),

    ("core_plus_lymphoid", "reference"):
        (42_593, 4),

    ("core_plus_lymphoid", "query"):
        (123_250, 4),

    ("extended_myeloid", "reference"):
        (135_538, 5),

    ("extended_myeloid", "query"):
        (123_250, 4),

    ("review_cycling", "reference"):
        (17_449, 2),

    ("review_cycling", "query"):
        (38_634, 1),

    ("review_ambiguous", "reference"):
        (17_449, 2),

    ("review_ambiguous", "query"):
        (18_667, 2),
}


observed_composition_core = {
    (
        str(
            row[
                "strategy"
            ]
        ),
        str(
            row[
                "cnv_group"
            ]
        ),
    ):
        (
            int(
                row[
                    "n_cells"
                ]
            ),
            int(
                row[
                    "n_clusters"
                ]
            ),
        )

    for _, row
    in composition_summary.iterrows()
}


if observed_composition_core != expected_composition_core:
    raise RuntimeError(
        "Strategy composition cell/cluster counts differ "
        "from the archived execution."
    )


print(
    "[HISTORICAL STRATEGY COMPOSITION COUNTS REPRODUCED]"
)

display(
    composition_summary
)

[SAVED] G:\Repository\glioma-cnv-single-cell-atlas\metadata\CNV_strategy_composition_summary_HVG8000_r0_6.tsv shape=(10, 20)
[HISTORICAL STRATEGY COMPOSITION COUNTS REPRODUCED]


,strategy,cnv_group,n_cells,n_clusters,top_core_dataset,top_core_dataset_n,top_core_dataset_fraction,n_core_dataset,top_condition,top_condition_n,top_condition_fraction,n_condition,top_patient_id,top_patient_id_n,top_patient_id_fraction,n_patient_id,top_sample_id,top_sample_id_n,top_sample_id_fraction,n_sample_id
0,core_conservative,reference,17449,2,DS4_GSE278450,8301,0.475729,4,GBM,10071,0.577168,4,04,2753,0.157774,49,GSM8546845_BT396,1680,0.096281,80
1,core_conservative,query,123250,4,DS3_GSE182109,57663,0.467854,4,GBM,65587,0.532146,4,01,16024,0.130012,50,GSM8546834_BT363,7881,0.063943,80
2,core_plus_lymphoid,reference,42593,4,DS3_GSE182109,31720,0.744723,4,rGBM,19117,0.448830,4,02,14581,0.342333,50,GSM5518615_rGBM-02-5,5089,0.119480,83
3,core_plus_lymphoid,query,123250,4,DS3_GSE182109,57663,0.467854,4,GBM,65587,0.532146,4,01,16024,0.130012,50,GSM8546834_BT363,7881,0.063943,80
4,extended_myeloid,reference,135538,5,DS3_GSE182109,88347,0.651825,4,GBM,47191,0.348175,4,01,29074,0.214508,50,GSM5518597_rGBM-01-B,9183,0.067752,83
5,extended_myeloid,query,123250,4,DS3_GSE182109,57663,0.467854,4,GBM,65587,0.532146,4,01,16024,0.130012,50,GSM8546834_BT363,7881,0.063943,80
6,review_cycling,reference,17449,2,DS4_GSE278450,8301,0.475729,4,GBM,10071,0.577168,4,04,2753,0.157774,49,GSM8546845_BT396,1680,0.096281,80
7,review_cycling,query,38634,1,DS3_GSE182109,21352,0.552674,4,GBM,17282,0.447326,4,04,9286,0.240358,48,GSM5518621_rGBM-04-3,2210,0.057203,75
8,review_ambiguous,reference,17449,2,DS4_GSE278450,8301,0.475729,4,GBM,10071,0.577168,4,04,2753,0.157774,49,GSM8546845_BT396,1680,0.096281,80
9,review_ambiguous,query,18667,2,DS3_GSE182109,17636,0.944769,4,ndGBM,14241,0.762897,4,10,9654,0.517169,50,GSM5518629_ndGBM-10,9654,0.517169,81


In [10]:
# =========================
# 9) FINAL OUTPUT MANIFEST
# =========================

manifest_items = [
    (
        "readiness_report",
        READINESS_TSV,
    ),
    (
        "cluster_category_counts",
        CLUSTER_CATEGORY_TSV,
    ),
    (
        "cell_category_counts",
        CELL_CATEGORY_TSV,
    ),
    (
        "strategy_definitions",
        STRATEGY_TSV,
    ),
    (
        "gene_position_report",
        GENE_POSITION_REPORT_TSV,
    ),
    (
        "gene_order_header",
        GENE_MATCH_TSV,
    ),
    (
        "gene_order_noheader",
        GENE_ORDER_NOHEADER_TSV,
    ),
    (
        "unmatched_genes",
        GENE_UNMATCHED_TSV,
    ),
    (
        "allcells_fullgene_countsX_h5ad",
        ALLCELLS_H5AD,
    ),
    (
        "allcells_gene_ordered_countsX_h5ad",
        GENE_ORDERED_H5AD,
    ),
    (
        "strategy_input_manifest",
        STRATEGY_MANIFEST_TSV,
    ),
    (
        "reference_group_names_summary",
        REF_GROUPS_TSV,
    ),
    (
        "strategy_composition_summary",
        COMPOSITION_TSV,
    ),
]


for _, row in strategy_manifest.iterrows():

    manifest_items.append(
        (
            f"annotation_{row['strategy']}",
            Path(
                row[
                    "annotation_tsv"
                ]
            ),
        )
    )

    manifest_items.append(
        (
            f"sampled_h5ad_{row['strategy']}",
            Path(
                row[
                    "sampled_h5ad"
                ]
            ),
        )
    )


manifest = pd.DataFrame(
    [
        {
            "item":
                name,

            "path":
                str(
                    path
                ),

            "exists":
                Path(
                    path
                ).exists(),

            "size_mb":
                (
                    round(
                        Path(
                            path
                        ).stat().st_size
                        / (
                            1024 ** 2
                        ),
                        3,
                    )
                    if Path(
                        path
                    ).exists()
                    else None
                ),
        }
        for name, path
        in manifest_items
    ]
)


MANIFEST_TSV = (
    METADATA_DIR
    / "CNV_input_output_manifest_HVG8000_r0_6.tsv"
)


write_tsv_replace(
    manifest,
    MANIFEST_TSV,
)


if manifest.shape != EXPECTED_MANIFEST_SHAPE:
    raise AssertionError(
        "Historical output-manifest shape changed."
    )


if not manifest[
    "exists"
].all():
    missing_outputs = (
        manifest.loc[
            ~manifest[
                "exists"
            ],
            [
                "item",
                "path",
            ],
        ]
    )

    raise RuntimeError(
        "CNV input preparation has missing outputs:\n"
        + missing_outputs.to_string(
            index=False
        )
    )


display(
    manifest
)


print(
    "\nDONE — canonical historical CNV-input preparation "
    "reproduced with memory-safe execution."
)

print(
    "Readiness:",
    readiness.shape,
)

print(
    "Cluster category counts:",
    cluster_role_counts.shape,
)

print(
    "Cell category counts:",
    role_cell_counts.shape,
)

print(
    "Gene-order table:",
    infercnv_gene_order.shape,
)

print(
    "All-cell full-gene:",
    (
        EXPECTED_N_CELLS,
        EXPECTED_N_GENES,
    ),
)

print(
    "All-cell gene-ordered:",
    (
        EXPECTED_N_CELLS,
        EXPECTED_GENE_ORDERED_GENES,
    ),
)

print(
    "Strategy manifest:",
    strategy_manifest.shape,
)

print(
    "Composition summary:",
    composition_summary.shape,
)

print(
    "Manifest:",
    manifest.shape,
)

print(
    "\nScratch retained for restart safety:"
)

print(
    SCRATCH_ROOT
)

[SAVED] G:\Repository\glioma-cnv-single-cell-atlas\metadata\CNV_input_output_manifest_HVG8000_r0_6.tsv shape=(23, 4)


,item,path,exists,size_mb
0,readiness_report,G:\Repository\glioma-cnv-single-cell-atlas\met...,True,0.001
1,cluster_category_counts,G:\Repository\glioma-cnv-single-cell-atlas\met...,True,0.002
2,cell_category_counts,G:\Repository\glioma-cnv-single-cell-atlas\met...,True,0.001
3,strategy_definitions,G:\Repository\glioma-cnv-single-cell-atlas\met...,True,0.001
4,gene_position_report,G:\Repository\glioma-cnv-single-cell-atlas\met...,True,0.000
5,gene_order_header,G:\Repository\glioma-cnv-single-cell-atlas\met...,True,0.410
6,gene_order_noheader,G:\Repository\glioma-cnv-single-cell-atlas\met...,True,0.410
7,unmatched_genes,G:\Repository\glioma-cnv-single-cell-atlas\met...,True,0.001
8,allcells_fullgene_countsX_h5ad,G:\Repository\glioma-cnv-single-cell-atlas\dat...,True,1699.716
9,allcells_gene_ordered_countsX_h5ad,G:\Repository\glioma-cnv-single-cell-atlas\dat...,True,1742.361



DONE — canonical historical CNV-input preparation reproduced with memory-safe execution.
Readiness: (1, 15)
Cluster category counts: (18, 7)
Cell category counts: (9, 3)
Gene-order table: (15163, 4)
All-cell full-gene: (351427, 15176)
All-cell gene-ordered: (351427, 15163)
Strategy manifest: (5, 12)
Composition summary: (10, 20)
Manifest: (23, 4)

Scratch retained for restart safety:
E:\glioma-cnv-scratch\main_analysis_10


## Expected successful completion

```text
Readiness:                (1, 15)
Cluster category counts:  (18, 7)
Cell category counts:     (9, 3)
Gene-order table:         (15163, 4)

All-cell full-gene:       (351427, 15176)
All-cell gene-ordered:    (351427, 15163)

Strategy manifest:        (5, 12)
Composition summary:      (10, 20)
Manifest:                 (23, 4)
```

Expected sampled H5AD dimensions:

```text
core_conservative     (32640, 15163)
core_plus_lymphoid    (38807, 15163)
extended_myeloid      (45000, 15163)
review_cycling        (14640, 15163)
review_ambiguous      (19443, 15163)
```

Primary gene-order checkpoint:

```text
15,176 / 15,176 genes matched to GENCODE v49
15,163 canonical chr1–22/X/Y genes retained
13 noncanonical mitochondrial genes excluded
```

The scratch directory can be retained until notebook `11_run_infercnvpy` has
validated all five sampled inputs.